# AIST-FYP Colab Wikipedia Preprocessing

This notebook preprocesses Wikipedia corpus data on Colab by reusing the project scripts:
- `scripts/download_wikipedia.py`
- `scripts/prepare_wikipedia_chunks.py`
- `scripts/generate_embeddings.py`
- `scripts/build_faiss_index.py`
- `scripts/build_bm25_index.py`

Default mode is **production** with **FAISS + BM25** and uses a **Google Drive persistent work root** so checkpoints survive Colab runtime resets.

‚ú?Recent update: embedding generation now uses a **low-memory streaming path** (JSONL two-pass: count first, then batch encode) and writes directly to `.npy` with **manifest-only checkpoint resume**. This is designed for very large chunk files.

‚ú?Recent update: FAISS/BM25 build cells now expose **low-memory tuning knobs** (`--add-batch-size`, `--tokenize-batch-size`, `--spacy-pipe-batch-size`) to reduce OOM risk on large corpora.

‚ú?Recent update: manual interrupt in FAISS/BM25 build cells now triggers a **checkpoint snapshot sync to Drive** before the cell exits, so you can recover from the latest committed checkpoint state.

## üîë Optional Secrets

If needed, add secrets in Colab sidebar (üîë):
- `HUGGINGFACE_TOKEN` (optional)

This preprocessing flow does not require OpenAI/DeepSeek keys.

In [ ]:
# ==============================
# Parameters (edit this cell)
# ==============================
REPO_URL = "https://github.com/xiashuidaolaoshuren/AIST-FYP.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/AIST-FYP"
COLAB_ENV_PROJECT = "colab/env"
COLAB_UV_EXTRAS = ["preprocessing"]

# Processing strategy
STRATEGY = "production"  # development | validation | production
DUMP_DATE = "latest"     # only used for production
MAX_ARTICLES_OVERRIDE = None  # for dev/validation quick runs, e.g., 5000

# Runtime/storage policy
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/AIST-FYP-colab-preprocess"
LOCAL_WORK_ROOT = f"{DRIVE_OUTPUT_ROOT}/work"  # Persistent workspace to survive Colab runtime resets
RUN_TAG = "wiki_preprocess_production"
REUSE_EXISTING_ARTICLE_JSONL = True  # production: skip source download if intermediate exists and reset is off
SKIP_EXPORT_COPY_WHEN_PERSISTENT = True  # if LOCAL_WORK_ROOT is on Drive, avoid duplicate artifact copy in Step 6

# Build targets
BUILD_FAISS = True
BUILD_BM25 = True

# FAISS settings
FAISS_INDEX_TYPE = "IVFFLAT"  # FLAT | IVFFLAT | HNSW
FAISS_NLIST = 4096
FAISS_NPROBE = 128
FAISS_HNSW_M = 32
FAISS_USE_GPU = False  # Force CPU FAISS in Colab config due faiss-gpu compatibility limits
FAISS_GPU_ID = 0      # Kept for compatibility; ignored when FAISS_USE_GPU=False
FAISS_NO_PROGRESS = False  # Set True for cleaner logs/CI

# Checkpoint behavior
RESUME = True
RESET_CHECKPOINT = False
CHUNKING_CHECKPOINT_INTERVAL = 1000
EMBEDDING_CHECKPOINT_INTERVAL = 10000  # Streaming embedding resume uses manifest-only progress
FAISS_CHECKPOINT_INTERVAL = 200000
FAISS_ADD_BATCH_SIZE = 50000  # Reduce (e.g., 10000) if host RAM is tight
BM25_CHECKPOINT_INTERVAL = 5000

# BM25 low-memory tokenization knobs
BM25_TOKENIZE_BATCH_SIZE = 2048     # Lower value => less RAM, more overhead
BM25_SPACY_PIPE_BATCH_SIZE = 256    # Lower value => less RAM, slower
BM25_NO_PROGRESS = False            # Set True for cleaner logs/CI

# Performance knobs
EMBED_BATCH_SIZE_OVERRIDE = None  # Set smaller value (e.g., 64/128) if GPU OOM
DISABLE_FP16 = False
INSTALL_SPACY_MODEL = True

In [ ]:
import os
import json
import shlex
import shutil
import subprocess
import yaml
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

def shell_join(parts):
    return " ".join(shlex.quote(str(p)) for p in parts)

def _sync_interrupt_checkpoint(cmd, on_interrupt):
    if on_interrupt is not None:
        on_interrupt()
        return

    cmd_str = str(cmd)
    if "scripts/build_faiss_index.py" in cmd_str:
        sync_checkpoint_snapshot("faiss_interrupt")
    elif "scripts/build_bm25_index.py" in cmd_str:
        sync_checkpoint_snapshot("bm25_interrupt")

def run(cmd, cwd=None, check=True, stream=True, on_interrupt=None):
    print(f"\n$ {cmd}")
    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        out_lines = []
        assert process.stdout is not None
        try:
            for line in process.stdout:
                print(line, end="")
                out_lines.append(line)
            process.wait()
        except KeyboardInterrupt:
            print("\n‚ö†Ô∏è Interrupted by user.")
            try:
                _sync_interrupt_checkpoint(cmd, on_interrupt)
            except Exception as sync_exc:
                print(f"‚ö†Ô∏è Checkpoint sync on interrupt failed: {sync_exc}")
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise

        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout="".join(out_lines),
            stderr=None,
        )
    else:
        try:
            completed = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        except KeyboardInterrupt:
            print("\n‚ö†Ô∏è Interrupted by user.")
            try:
                _sync_interrupt_checkpoint(cmd, on_interrupt)
            except Exception as sync_exc:
                print(f"‚ö†Ô∏è Checkpoint sync on interrupt failed: {sync_exc}")
            raise
        if completed.stdout:
            print(completed.stdout)

    if completed.returncode != 0:
        if not stream and completed.stderr:
            print(completed.stderr)
        if check:
            raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def ensure_symlink_dir(link_path: Path, target_path: Path):
    target_path.mkdir(parents=True, exist_ok=True)
    if link_path.exists() or link_path.is_symlink():
        if link_path.is_symlink() and link_path.resolve() == target_path.resolve():
            return
        if link_path.is_symlink() or link_path.is_file():
            link_path.unlink()
        else:
            shutil.rmtree(link_path)
    link_path.parent.mkdir(parents=True, exist_ok=True)
    link_path.symlink_to(target_path, target_is_directory=True)
    return link_path, target_path, "linked"

def _list_files_with_size(root: Path) -> tuple[list[Path], int]:
    files = []
    total_bytes = 0
    for path in root.rglob("*"):
        if path.is_file():
            files.append(path)
            try:
                total_bytes += path.stat().st_size
            except OSError:
                pass
    return files, total_bytes

def _copy_tree_with_progress(src: Path, dst: Path):
    files, total_bytes = _list_files_with_size(src)
    chunk_size = 1024 * 1024  # 1MB chunks for stable progress updates

    dst.mkdir(parents=True, exist_ok=True)

    # Recreate directory tree first so copy order is deterministic.
    for dir_path in sorted((p for p in src.rglob("*") if p.is_dir()), key=lambda p: len(p.parts)):
        rel = dir_path.relative_to(src)
        target_dir = dst / rel
        target_dir.mkdir(parents=True, exist_ok=True)

    with tqdm(total=total_bytes, unit="B", unit_scale=True, desc=f"Sync {src.name}") as progress_bar:
        for file_path in files:
            rel = file_path.relative_to(src)
            target_file = dst / rel
            target_file.parent.mkdir(parents=True, exist_ok=True)

            with file_path.open("rb") as fsrc, target_file.open("wb") as fdst:
                while True:
                    chunk = fsrc.read(chunk_size)
                    if not chunk:
                        break
                    fdst.write(chunk)
                    progress_bar.update(len(chunk))

            try:
                shutil.copystat(file_path, target_file)
            except OSError:
                pass

def sync_checkpoint_snapshot(reason: str):
    checkpoint_root = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints"
    if not checkpoint_root.exists():
        print(f"No checkpoint root found to sync: {checkpoint_root}")
        return None

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    snapshot_root = Path(DRIVE_OUTPUT_ROOT) / "interrupt_checkpoints"
    destination = snapshot_root / f"{RUN_TAG}_{STRATEGY}_{reason}_{timestamp}"
    copied_dirs = []

    for name in ("faiss", "bm25"):
        src = checkpoint_root / name
        if src.exists():
            dst = destination / name
            dst.parent.mkdir(parents=True, exist_ok=True)
            _copy_tree_with_progress(src, dst)
            copied_dirs.append(str(dst))

    if not copied_dirs:
        print(f"No FAISS/BM25 checkpoint directories to sync under: {checkpoint_root}")
        return None

    manifest_path = destination / "snapshot_manifest.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(
        json.dumps(
            {
                "timestamp": timestamp,
                "reason": reason,
                "strategy": STRATEGY,
                "checkpoint_root": str(checkpoint_root),
                "copied_dirs": copied_dirs,
            },
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )

    print(f"‚ú?Checkpoint snapshot synced to: {destination}")
    return destination

In [ ]:
# Mount Drive + clone/update repo + install dependencies
from google.colab import drive
drive.mount('/content/drive')

if not str(LOCAL_WORK_ROOT).startswith('/content/drive/'):
    print(f"‚ö†Ô∏è LOCAL_WORK_ROOT is not on Drive: {LOCAL_WORK_ROOT}")
    print("Checkpoint persistence across runtime resets may be lost.")

ensure_dir(LOCAL_WORK_ROOT)
print("Persistent LOCAL_WORK_ROOT:", LOCAL_WORK_ROOT)
print("Drive output root:", DRIVE_OUTPUT_ROOT)

repo_path = Path(REPO_DIR)
if repo_path.exists():
    print(f"Repo exists: {repo_path}")
    run("git fetch --all", cwd=REPO_DIR)
    run(f"git checkout {REPO_BRANCH}", cwd=REPO_DIR)
    run(f"git pull origin {REPO_BRANCH}", cwd=REPO_DIR, check=False)
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

run("git rev-parse --abbrev-ref HEAD", cwd=REPO_DIR)
run("git log -1 --oneline", cwd=REPO_DIR)

run("python -m pip install -U pip wheel setuptools", stream=True)
run("python -m pip install -U uv", stream=True)

uv_project = Path(REPO_DIR) / COLAB_ENV_PROJECT
extras_args = " ".join(f"--extra {extra}" for extra in COLAB_UV_EXTRAS)
sync_cmd = f"uv sync --project {uv_project} {extras_args}"
result = run(sync_cmd, cwd=REPO_DIR, check=False, stream=True)

spacy_model_wheel = "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
spacy_install_cmd = "python -m spacy download en_core_web_sm"

if result.returncode == 0:
    uv_python = uv_project / ".venv" / "bin" / "python"
    os.environ["PATH"] = f"{uv_python.parent}:{os.environ.get('PATH', '')}"
    spacy_install_cmd = f"uv pip install --python {uv_python} {spacy_model_wheel}"
    print(f"‚ú?uv sync complete: {uv_project}")
else:
    print('\n‚ö†Ô∏è uv sync failed. Falling back to pip requirements install...')

    # Colab-safe pip fallback for torch +cu121 pins
    requirements_path = Path(REPO_DIR) / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = f"pip install --extra-index-url {pytorch_index} -r {requirements_path}"
    fallback_result = run(install_cmd, cwd=REPO_DIR, check=False, stream=True)

    if fallback_result.returncode != 0:
        print('\n‚ö†Ô∏è Full requirements install failed. Falling back to Colab-torch-compatible install...')
        filtered = []
        skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==')
        for raw in requirements_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if any(line.startswith(prefix) for prefix in skip_prefixes):
                continue
            filtered.append(line)

        temp_req = Path(REPO_DIR) / 'requirements.colab.filtered.txt'
        temp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')

        run('python - <<"PY"\nimport torch\nimport torchvision\nimport torchaudio\nprint("torch", torch.__version__)\nprint("torchvision", torchvision.__version__)\nprint("torchaudio", torchaudio.__version__)\nPY', cwd=REPO_DIR)
        run(f"pip install -r {temp_req}", cwd=REPO_DIR, stream=True)

if INSTALL_SPACY_MODEL:
    run(spacy_install_cmd, cwd=REPO_DIR, stream=True)

# Quick CLI sanity checks
run("python scripts/download_wikipedia.py --help", cwd=REPO_DIR)
run("python scripts/prepare_wikipedia_chunks.py --help", cwd=REPO_DIR)
run("python scripts/generate_embeddings.py --help", cwd=REPO_DIR)
run("python scripts/build_faiss_index.py --help", cwd=REPO_DIR)
run("python scripts/build_bm25_index.py --help", cwd=REPO_DIR)

In [ ]:
# Fail fast if no GPU
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. In Colab: Runtime -> Change runtime type -> GPU")

print("GPU:", torch.cuda.get_device_name(0))
print("GPU count:", torch.cuda.device_count())

In [ ]:
# Build colab preprocessing config with persistent LOCAL_WORK_ROOT paths
base_config_path = Path(REPO_DIR) / "config.yaml"
colab_config_path = Path(REPO_DIR) / "config.colab.preprocess.yaml"

with open(base_config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

local_data = Path(LOCAL_WORK_ROOT) / "data"
cfg.setdefault("data", {})
cfg["data"]["wikipedia_dump"] = str(local_data / "raw" / "enwiki-latest-pages-articles.xml.bz2")
cfg["data"]["wikipedia_sample_dev"] = str(local_data / "raw" / "wiki_sample_development.jsonl")
cfg["data"]["wikipedia_sample_val"] = str(local_data / "raw" / "wiki_sample_validation.jsonl")
cfg["data"]["processed_chunks"] = str(local_data / "processed" / "wiki_chunks_{strategy}.jsonl")
cfg["data"]["embeddings"] = str(local_data / "embeddings" / "wiki_embeddings_{strategy}.npy")
cfg["data"]["embeddings_metadata"] = str(local_data / "embeddings" / "metadata_{strategy}.json")
cfg["data"]["faiss_index"] = str(local_data / "indexes" / "{strategy}" / "faiss.index")
cfg["data"]["index_metadata"] = str(local_data / "indexes" / "{strategy}" / "metadata.pkl")
cfg["data"]["index_config"] = str(local_data / "indexes" / "{strategy}" / "index_config.json")
cfg["data"]["bm25_index"] = str(local_data / "indexes" / "{strategy}" / "bm25_index.pkl")

cfg.setdefault("processing", {})["device"] = "cuda"
cfg.setdefault("verification", {}).setdefault("nli", {})["device"] = "cuda"
cfg.setdefault("verification", {}).setdefault("self_agreement", {})["device"] = "cuda"

cfg.setdefault("retrieval", {})
cfg["retrieval"].setdefault("faiss", {})["use_gpu"] = bool(FAISS_USE_GPU)
cfg["retrieval"]["faiss"]["gpu_id"] = int(FAISS_GPU_ID)

cfg.setdefault("checkpointing", {})
cfg["checkpointing"]["checkpoint_dir"] = str(local_data / "checkpoints" / "embeddings")
cfg["checkpointing"].setdefault("chunking", {})["checkpoint_dir"] = str(local_data / "checkpoints" / "chunking")
cfg["checkpointing"].setdefault("faiss", {})["checkpoint_dir"] = str(local_data / "checkpoints" / "faiss")
cfg["checkpointing"].setdefault("bm25", {})["checkpoint_dir"] = str(local_data / "checkpoints" / "bm25")

ensure_dir(local_data / "raw")
ensure_dir(local_data / "processed")
ensure_dir(local_data / "embeddings")
ensure_dir(local_data / "indexes")
ensure_dir(local_data / "checkpoints")
ensure_dir(local_data / "checkpoints" / "embeddings")

with open(colab_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("Wrote config:", colab_config_path)
print("Local work root:", LOCAL_WORK_ROOT)
print("Strategy:", STRATEGY)
print("Chunk checkpoint dir:", cfg["checkpointing"]["chunking"]["checkpoint_dir"])
print("FAISS checkpoint dir:", cfg["checkpointing"]["faiss"]["checkpoint_dir"])
print("BM25 checkpoint dir:", cfg["checkpointing"]["bm25"]["checkpoint_dir"])
print("Embedding checkpoint dir:", cfg["checkpointing"]["checkpoint_dir"])

In [ ]:
# Step 1: Download Wikipedia source data (optional if intermediate article JSONL exists)
article_jsonl = str(Path(LOCAL_WORK_ROOT) / "data" / "processed" / f"wiki_articles_{STRATEGY}.jsonl")
article_jsonl_path = Path(article_jsonl)
chunk_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints" / "chunking"

print(f"Step 1 checkpoint context: chunking checkpoints at {chunk_ckpt_dir}")

should_download_source = True
if STRATEGY == "production" and REUSE_EXISTING_ARTICLE_JSONL and article_jsonl_path.exists() and not RESET_CHECKPOINT:
    should_download_source = False
    print(f"Skipping Step 1 download: reusing intermediate article JSONL at {article_jsonl_path}")

if should_download_source:
    download_cmd = [
        "python",
        "scripts/download_wikipedia.py",
        "--strategy", STRATEGY,
        "--config", "config.colab.preprocess.yaml",
    ]

    if STRATEGY == "production":
        download_cmd += ["--dump-date", DUMP_DATE]
    elif MAX_ARTICLES_OVERRIDE is not None:
        download_cmd += ["--max-articles", str(MAX_ARTICLES_OVERRIDE)]

    run(shell_join(download_cmd), cwd=REPO_DIR, stream=True)

In [ ]:
# Step 2: Prepare sentence-level chunks
article_jsonl = str(Path(LOCAL_WORK_ROOT) / "data" / "processed" / f"wiki_articles_{STRATEGY}.jsonl")
chunk_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints" / "chunking"
print(f"Step 2 checkpoint dir: {chunk_ckpt_dir}")

chunk_cmd = [
    "python",
    "scripts/prepare_wikipedia_chunks.py",
    "--strategy", STRATEGY,
    "--config", "config.colab.preprocess.yaml",
    "--article-jsonl", article_jsonl,
    "--checkpoint-interval", str(CHUNKING_CHECKPOINT_INTERVAL),
]

if RESUME:
    chunk_cmd.append("--resume")
else:
    chunk_cmd.append("--no-resume")

if RESET_CHECKPOINT:
    chunk_cmd.append("--reset-checkpoint")

run(shell_join(chunk_cmd), cwd=REPO_DIR, stream=True)

In [ ]:
# Step 3: Generate embeddings (low-memory streaming mode)
embedding_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints" / "embeddings"
print(f"Step 3 checkpoint dir: {embedding_ckpt_dir}")
print("Step 3 note: generator performs JSONL pre-count + streaming batch encode to reduce RAM usage.")

embed_cmd = [
    "python",
    "scripts/generate_embeddings.py",
    "--strategy", STRATEGY,
    "--config", "config.colab.preprocess.yaml",
    "--device", "cuda",
    "--checkpoint-interval", str(EMBEDDING_CHECKPOINT_INTERVAL),
]

if EMBED_BATCH_SIZE_OVERRIDE is not None:
    embed_cmd += ["--batch-size", str(EMBED_BATCH_SIZE_OVERRIDE)]

if DISABLE_FP16:
    embed_cmd.append("--no-fp16")

run(shell_join(embed_cmd), cwd=REPO_DIR, stream=True)

In [ ]:
# Step 4: Build FAISS index
faiss_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints" / "faiss"
print(f"Step 4 checkpoint dir: {faiss_ckpt_dir}")
print(f"Step 4 FAISS GPU config: enabled={FAISS_USE_GPU}, gpu_id={FAISS_GPU_ID}")
print("Step 4 note: reduce FAISS_ADD_BATCH_SIZE (e.g., 10000) if host RAM OOM occurs.")

if BUILD_FAISS:
    faiss_cmd = [
        "python",
        "scripts/build_faiss_index.py",
        "--strategy", STRATEGY,
        "--config", "config.colab.preprocess.yaml",
        "--index-type", FAISS_INDEX_TYPE,
        "--nlist", str(FAISS_NLIST),
        "--nprobe", str(FAISS_NPROBE),
        "--hnsw-m", str(FAISS_HNSW_M),
        "--checkpoint-interval", str(FAISS_CHECKPOINT_INTERVAL),
        "--add-batch-size", str(FAISS_ADD_BATCH_SIZE),
    ]

    # GPU mode now defaults from config (retrieval.faiss.use_gpu/gpu_id).
    # Uncomment to force override per run:
    # faiss_cmd += ["--use-gpu", "--gpu-id", str(FAISS_GPU_ID)]
    # faiss_cmd += ["--no-use-gpu"]

    if FAISS_NO_PROGRESS:
        faiss_cmd.append("--no-progress")

    if RESUME:
        faiss_cmd.append("--resume")
    else:
        faiss_cmd.append("--no-resume")

    if RESET_CHECKPOINT:
        faiss_cmd.append("--reset-checkpoint")

    run(shell_join(faiss_cmd), cwd=REPO_DIR, stream=True)
else:
    print("BUILD_FAISS=False, skipped.")

In [ ]:
# Step 5: Build BM25 index
bm25_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints" / "bm25"
print(f"Step 5 checkpoint dir: {bm25_ckpt_dir}")
print(
    "Step 5 note: lower BM25_TOKENIZE_BATCH_SIZE and BM25_SPACY_PIPE_BATCH_SIZE "
    "if host RAM OOM occurs. Example: 512 / 128."
)

if BUILD_BM25:
    bm25_cmd = [
        "python",
        "scripts/build_bm25_index.py",
        "--strategy", STRATEGY,
        "--config", "config.colab.preprocess.yaml",
        "--checkpoint-interval", str(BM25_CHECKPOINT_INTERVAL),
        "--tokenize-batch-size", str(BM25_TOKENIZE_BATCH_SIZE),
        "--spacy-pipe-batch-size", str(BM25_SPACY_PIPE_BATCH_SIZE),
    ]

    if BM25_NO_PROGRESS:
        bm25_cmd.append("--no-progress")

    if RESUME:
        bm25_cmd.append("--resume")
    else:
        bm25_cmd.append("--no-resume")

    if RESET_CHECKPOINT:
        bm25_cmd.append("--reset-checkpoint")

    run(shell_join(bm25_cmd), cwd=REPO_DIR, stream=True)
else:
    print("BUILD_BM25=False, skipped.")

In [ ]:
# Step 6: Validate artifacts and export manifest/snapshot
with open(Path(REPO_DIR) / "config.colab.preprocess.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

expected = [
    Path(cfg["data"]["processed_chunks"].format(strategy=STRATEGY)),
    Path(cfg["data"]["embeddings"].format(strategy=STRATEGY)),
    Path(cfg["data"]["embeddings_metadata"].format(strategy=STRATEGY)),
    Path(cfg["data"]["faiss_index"].format(strategy=STRATEGY)),
    Path(cfg["data"]["index_metadata"].format(strategy=STRATEGY)),
]

if BUILD_BM25:
    expected.append(Path(cfg["data"]["bm25_index"].format(strategy=STRATEGY)))

missing = [str(p) for p in expected if not p.exists()]
if missing:
    raise FileNotFoundError("Missing expected artifacts:\n" + "\n".join(missing))

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_root = Path(DRIVE_OUTPUT_ROOT) / f"{RUN_TAG}_{STRATEGY}_{run_stamp}"
ensure_dir(export_root)

local_work_root_path = Path(LOCAL_WORK_ROOT).resolve()
artifacts_on_drive = str(local_work_root_path).startswith('/content/drive/')

exported_paths = []
if artifacts_on_drive and SKIP_EXPORT_COPY_WHEN_PERSISTENT:
    print("Artifacts are already on Drive; skipping duplicate file copy.")
    print("Writing run manifest only.")
    exported_paths = [str(p) for p in expected]
else:
    for p in expected:
        rel = p.relative_to(local_work_root_path) if str(p).startswith(str(local_work_root_path)) else Path(p.name)
        dest = export_root / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
        exported_paths.append(str(dest))
        print("Exported:", p, "->", dest)

manifest = {
    "timestamp": run_stamp,
    "strategy": STRATEGY,
    "dump_date": DUMP_DATE,
    "build_faiss": BUILD_FAISS,
    "build_bm25": BUILD_BM25,
    "local_work_root": LOCAL_WORK_ROOT,
    "repo_dir": REPO_DIR,
    "export_root": str(export_root),
    "artifacts": [str(p) for p in expected],
    "exported_artifacts": exported_paths,
    "artifacts_already_on_drive": artifacts_on_drive,
    "skipped_copy_when_persistent": bool(artifacts_on_drive and SKIP_EXPORT_COPY_WHEN_PERSISTENT),
}

manifest_path = export_root / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("\nDone. Manifest:", manifest_path)

## üì¶ Step 7: Package Huge Artifacts for Reliable Download

Use this section when artifacts are too large for stable browser download.

Default behavior:
- Per-artifact archives (one package per file)
- Gzip-compressed tar (`.tar.gz`)
- Split into **2 GB** parts
- SHA-256 checksum for **every part**

Recommended transfer method: **CLI resumable download** (`rclone` / `gsutil`).

In [ ]:
# Step 7A: Build split archives + SHA-256 checksums (per artifact)
import hashlib
import tarfile

# Packaging policy (editable)
PACKAGE_MIN_SIZE_GB = 1.0      # only package files >= this size
SPLIT_SIZE_GB = 2              # selected policy: 2 GB parts
DELETE_INTERMEDIATE_TAR = True # remove .tar.gz after splitting to save Drive space

with open(Path(REPO_DIR) / "config.colab.preprocess.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

strategy = STRATEGY
local_work_root = Path(LOCAL_WORK_ROOT).resolve()
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_root = Path(DRIVE_OUTPUT_ROOT) / f"{RUN_TAG}_{strategy}_{run_stamp}" / "download_packages"
ensure_dir(export_root)

artifact_candidates = [
    Path(cfg["data"]["embeddings"].format(strategy=strategy)),
    Path(cfg["data"]["faiss_index"].format(strategy=strategy)),
    Path(cfg["data"]["index_metadata"].format(strategy=strategy)),
    Path(cfg["data"]["bm25_index"].format(strategy=strategy)) if BUILD_BM25 else None,
]
artifact_candidates = [p for p in artifact_candidates if p is not None and p.exists()]

min_size_bytes = int(PACKAGE_MIN_SIZE_GB * (1024 ** 3))
split_size_bytes = int(SPLIT_SIZE_GB * (1024 ** 3))

targets = [p for p in artifact_candidates if p.stat().st_size >= min_size_bytes]
if not targets:
    print("No large artifacts matched packaging threshold.")
    print("Candidates:")
    for p in artifact_candidates:
        print(f"- {p} ({p.stat().st_size / (1024**3):.2f} GB)")
else:
    print(f"Packaging {len(targets)} artifact(s) to: {export_root}")

package_manifest = {
    "timestamp": run_stamp,
    "strategy": strategy,
    "export_root": str(export_root),
    "split_size_bytes": split_size_bytes,
    "minimum_size_bytes": min_size_bytes,
    "artifacts": [],
}

def sha256_of_file(file_path: Path) -> str:
    h = hashlib.sha256()
    with open(file_path, "rb") as fp:
        while True:
            chunk = fp.read(8 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

for src in targets:
    artifact_name = src.name
    safe_name = artifact_name.replace("/", "_").replace("\\", "_")
    artifact_dir = export_root / safe_name
    ensure_dir(artifact_dir)

    tar_path = artifact_dir / f"{artifact_name}.tar.gz"
    print(f"\n[Pack] {src} -> {tar_path}")
    with tarfile.open(tar_path, "w:gz") as tar:
        tar.add(src, arcname=artifact_name)

    part_paths = []
    with open(tar_path, "rb") as rf:
        part_idx = 1
        while True:
            block = rf.read(split_size_bytes)
            if not block:
                break
            part_path = artifact_dir / f"{artifact_name}.tar.gz.part{part_idx:03d}"
            with open(part_path, "wb") as wf:
                wf.write(block)
            part_paths.append(part_path)
            part_idx += 1

    checksums_path = artifact_dir / "checksums.sha256"
    with open(checksums_path, "w", encoding="utf-8") as cf:
        for part in part_paths:
            digest = sha256_of_file(part)
            cf.write(f"{digest}  {part.name}\n")

    if DELETE_INTERMEDIATE_TAR and tar_path.exists():
        tar_path.unlink()

    manifest_row = {
        "source_path": str(src),
        "source_size_bytes": src.stat().st_size,
        "archive_dir": str(artifact_dir),
        "parts": [p.name for p in part_paths],
        "part_sizes_bytes": [p.stat().st_size for p in part_paths],
        "checksums_file": str(checksums_path),
    }
    package_manifest["artifacts"].append(manifest_row)

manifest_path = export_root / "package_manifest.json"
manifest_path.write_text(json.dumps(package_manifest, indent=2), encoding="utf-8")

print("\nPackaging complete.")
print("Package manifest:", manifest_path)
for row in package_manifest["artifacts"]:
    total_parts = len(row["parts"])
    src_gb = row["source_size_bytes"] / (1024 ** 3)
    print(f"- {Path(row['source_path']).name}: {src_gb:.2f} GB -> {total_parts} part(s)")

## üöö Step 7B: Resumable CLI Download (Local Machine)

Run this local script for one-command workflow (download + verify + reconstruct):

- `python scripts/download_colab_packages.py --source "gdrive:AIST-FYP-colab-preprocess/<run_folder>/download_packages" --dest "<local_target_dir>/download_packages" --tool rclone`

Useful variants:
- Verify + reconstruct from already-downloaded files:
  - `python scripts/download_colab_packages.py --dest "<local_target_dir>/download_packages" --skip-download`
- Verify only:
  - `python scripts/download_colab_packages.py --dest "<local_target_dir>/download_packages" --skip-download --skip-reconstruct`

Fallback direct CLI (manual):
- `rclone copy --progress --retries 10 --low-level-retries 20 --checksum "gdrive:AIST-FYP-colab-preprocess/<run_folder>/download_packages" "<local_target_dir>"`

In [ ]:
# Step 7C: (Optional local helper) verify checksums + reconstruct tar.gz from parts
import json
import hashlib
import shutil
from pathlib import Path

print("Run this logic on your local machine after downloading parts.")
print("It is provided here as reference code.")

LOCAL_DOWNLOAD_ROOT = "<local_target_dir>/download_packages"  # edit locally before use

local_root = Path(LOCAL_DOWNLOAD_ROOT)
manifest_path = local_root / "package_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Missing package manifest: {manifest_path}")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

def sha256_of_file(file_path: Path) -> str:
    h = hashlib.sha256()
    with open(file_path, "rb") as fp:
        while True:
            chunk = fp.read(8 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

for artifact in manifest.get("artifacts", []):
    artifact_dir = Path(artifact["archive_dir"]).name
    artifact_local_dir = local_root / artifact_dir
    checksums_path = artifact_local_dir / "checksums.sha256"
    if not checksums_path.exists():
        raise FileNotFoundError(f"Missing checksums file: {checksums_path}")

    print(f"\n[Verify] {artifact_local_dir}")
    expected = []
    for line in checksums_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        digest, filename = line.split("  ", 1)
        expected.append((digest.strip(), filename.strip()))

    for digest, filename in expected:
        part_path = artifact_local_dir / filename
        if not part_path.exists():
            raise FileNotFoundError(f"Missing part: {part_path}")
        actual = sha256_of_file(part_path)
        if actual != digest:
            raise ValueError(f"Checksum mismatch: {part_path.name}")

    reconstructed = artifact_local_dir / f"{Path(artifact['source_path']).name}.tar.gz"
    with open(reconstructed, "wb") as out_f:
        for part_name in artifact["parts"]:
            part_path = artifact_local_dir / part_name
            with open(part_path, "rb") as in_f:
                shutil.copyfileobj(in_f, out_f, length=8 * 1024 * 1024)

    print(f"Reconstructed archive: {reconstructed}")

print("\nAll listed artifacts verified and reconstructed.")